### Maria Eduarda Selhorst
### Yasmin Trembulack Agostinho

## Funções

In [169]:
import numpy as np

In [170]:
def cadeia_markov(matriz, estado_inicial, periodo):
    estados = [estado_inicial] # Armazena estados ao longo do tempo
    if isinstance(periodo, str) or isinstance(periodo, list):
        periodo = len(periodo)
        
    for _ in range(periodo):
        # Multiplica a matriz de transição pelo estado atual (vetor)
        novo_estado = np.dot(estados[-1], matriz)
        estados.append(novo_estado) # Adiciona novo estado a lista
    # Converte para array para facilitar a manipulação
    estados = np.array(estados).round(3) 
    return estados

In [171]:
def print_markov_response(simulacao, nomes_estados, nome_intervalo):
    for i, estado in enumerate(simulacao):
        print(f"{nome_intervalo} {i}:")
        for j, probabilidade in enumerate(estado):
            print(f"  {nomes_estados[j]}: {probabilidade:.2f} ({probabilidade*100:.0f}%)")
        print()

In [172]:
def print_bayes_response(tabela, column_01, column_02, size: int, porcent = False):
    print(f"{column_01:<{size}} {column_02}")
    print("-" * (size + 20))
    for evento, prob in tabela:
        valor = prob * 100 if porcent else prob
        sufixo = "%" if porcent else ""
        print(f"{evento:<{size}} {valor:.2f}{sufixo}")
    print("-" * (size + 20))

## Exercício 01

In [173]:
# Manual - Caso de Zuriel
# Probabilidades iniciais
p_heresia = 0.05
p_n_heresia = 1 - p_heresia

# Probabilidades condicionais das evidências
p_rituais_heresia = 0.90
p_estrangeiros_heresia = 0.80
p_silencio_heresia = 0.75

p_rituais_n_heresia = 0.10
p_estrangeiros_n_heresia = 0.20
p_silencio_n_heresia = 0.25


In [174]:
# Probabilidade conjunta das evidências (assumindo independência condicional)
p_evidencias_heresia = p_rituais_heresia * p_estrangeiros_heresia * p_silencio_heresia
p_evidencias_n_heresia = p_rituais_n_heresia * p_estrangeiros_n_heresia * p_silencio_n_heresia

# Probabilidade total das evidências
p_evidencias = (p_evidencias_heresia * p_heresia) + (p_evidencias_n_heresia * p_n_heresia)

# Probabilidade final pelo Teorema de Bayes
p_heresia_dado_evidencias = (p_evidencias_heresia * p_heresia) / p_evidencias

In [175]:
# Impressão formatada
tabela = [
    ("P(Heresia)", p_heresia),
    ("P(Não Heresia)", p_n_heresia),
    ("P(Rituais | Heresia)", p_rituais_heresia),
    ("P(Estrangeiros | Heresia)", p_estrangeiros_heresia),
    ("P(Silêncio | Heresia)", p_silencio_heresia),
    ("P(Rituais | Não Heresia)", p_rituais_n_heresia),
    ("P(Estrangeiros | Não Heresia)", p_estrangeiros_n_heresia),
    ("P(Silêncio | Não Heresia)", p_silencio_n_heresia),
    ("P(R,E,S | Heresia)", p_evidencias_heresia),
    ("P(R,E,S | Não Heresia)", p_evidencias_n_heresia),
    ("P(R=sim,E=sim,S=sim)", p_evidencias),
    ("P(Heresia | R,E,S)", p_heresia_dado_evidencias)
]
print_bayes_response(tabela, 'Evento', 'Probabilidade', 40, True)

Evento                                   Probabilidade
------------------------------------------------------------
P(Heresia)                               5.00%
P(Não Heresia)                           95.00%
P(Rituais | Heresia)                     90.00%
P(Estrangeiros | Heresia)                80.00%
P(Silêncio | Heresia)                    75.00%
P(Rituais | Não Heresia)                 10.00%
P(Estrangeiros | Não Heresia)            20.00%
P(Silêncio | Não Heresia)                25.00%
P(R,E,S | Heresia)                       54.00%
P(R,E,S | Não Heresia)                   0.50%
P(R=sim,E=sim,S=sim)                     3.17%
P(Heresia | R,E,S)                       85.04%
------------------------------------------------------------


In [176]:
# Simulação de levitas observados
num_levitas = 1000

# Gerar estados reais (5% de heresia)
estados_reais = np.random.choice(
    ["Heresia", "Não Heresia"], 
    size=num_levitas,
    p=[p_heresia, p_n_heresia]
)

# Gerar evidências com base no estado real
evidencias = []
for estado in estados_reais:
    if estado == "Heresia":
        rituais = np.random.rand() < p_rituais_heresia
        estrangeiros = np.random.rand() < p_estrangeiros_heresia
        silencio = np.random.rand() < p_silencio_heresia
    else:
        rituais = np.random.rand() < p_rituais_n_heresia
        estrangeiros = np.random.rand() < p_estrangeiros_n_heresia
        silencio = np.random.rand() < p_silencio_n_heresia
    evidencias.append((rituais, estrangeiros, silencio))

In [177]:
# Classificar com base na probabilidade calculada
classificacoes = []
for r, e, s in evidencias:
    if r and e and s:  # Todas as evidências presentes
        prob = p_heresia_dado_evidencias
    else:
        # Aqui deveria ser calculada P(Heresia|evidências parciais)
        # Simplificando, usamos um valor menor
        prob = p_heresia_dado_evidencias * 0.3  # Ajuste empírico
   
    classificacoes.append("Heresia" if np.random.rand() < prob else "Não Heresia")

In [178]:
# Métricas
vp = sum(1 for real, classif in zip(estados_reais, classificacoes)
       if real == "Heresia" and classif == "Heresia")
fp = sum(1 for real, classif in zip(estados_reais, classificacoes)
       if real == "Não Heresia" and classif == "Heresia")

In [179]:
tabela = [
    ("Total de Levitas", num_levitas),
    ("Levitas com Heresia real", list(estados_reais).count('Heresia')),
    ("Levitas classificados como Heresia", classificacoes.count('Heresia')),
    ("Verdadeiros Positivos", vp),
    ("Falsos Positivos", fp),
]

print_bayes_response(
    tabela,
    'Probabilidade de heresia dado todas as evidências:',
    f'{p_heresia_dado_evidencias:.2%}',
    60
)

Probabilidade de heresia dado todas as evidências:           85.04%
--------------------------------------------------------------------------------
Total de Levitas                                             1000.00
Levitas com Heresia real                                     57.00
Levitas classificados como Heresia                           264.00
Verdadeiros Positivos                                        29.00
Falsos Positivos                                             235.00
--------------------------------------------------------------------------------


### Resposta:
A probabilidade de que Zuriel esteja de fato envolvido com heresia, dado todas as evidências, é aproximadamente 84.96%.
Conclusão: Mas Moisés deve considerar outras fontes de evidência, contexto histórico e talvez até um julgamento mais aprofundado antes de tomar uma decisão irreversível. Modelo é útil para detectar casos com forte suspeita, mas tem taxa de falsos positivos alta.


## Exercício 02

In [180]:
nomes_estados = ["Focado", "Eufórico", "Perigoso"]
# Eddie começa Focado
estado_inicial = np.array([1.0, 0.0, 0.0])
# Matriz de transição
matriz = np.array([
    [0.6, 0.3, 0.1],
    [0.4, 0.4, 0.2],
    [0.2, 0.5, 0.3]
])

In [181]:
periodos = int(input("Por favor, digite o número de intervalos entre músicas: "))
simulacao = cadeia_markov(matriz, estado_inicial, periodos)

In [182]:
print("\nDistribuição de probabilidade de Eddie ao longo dos intervalos:")
print_markov_response(
    simulacao=simulacao,
    nomes_estados=nomes_estados,
    nome_intervalo='Intervalo'
)


Distribuição de probabilidade de Eddie ao longo dos intervalos:
Intervalo 0:
  Focado: 1.00 (100%)
  Eufórico: 0.00 (0%)
  Perigoso: 0.00 (0%)

Intervalo 1:
  Focado: 0.60 (60%)
  Eufórico: 0.30 (30%)
  Perigoso: 0.10 (10%)

Intervalo 2:
  Focado: 0.50 (50%)
  Eufórico: 0.35 (35%)
  Perigoso: 0.15 (15%)

Intervalo 3:
  Focado: 0.47 (47%)
  Eufórico: 0.36 (36%)
  Perigoso: 0.17 (16%)



### Resposta:
Em média, Eddie vai ficar focado na maior parte do tempo.
No terceiro intervalo, a chance de ele estar no estado perigoso aumenta para 19%.
mas também passa uma boa parte eufórico, e às vezes entra em um estado perigoso.
Conclusão: Mesmo ele sendo focado na maioria das vezes, a chance de comportamento perigoso não é descartado. Pode ser bom alguém ficar de olho no Eddie durante a turnê.

## Exercício 03

In [183]:
nomes_estados = ["Lady Gaga", "Olivia Rodrigo", "Taylor Swift"]
# Olivia Rodrigo começa com a atenção da medía
estado_inicial = np.array([0.0, 1.0, 0.0])
# Matriz de transição
matriz = np.array([
    [0.6, 0.2, 0.3],
    [0.3, 0.5, 0.3],
    [0.1, 0.3, 0.4] 
])

In [184]:
dias = int(input("Digite o número de dias para simular: "))
simulacao = cadeia_markov(matriz, estado_inicial, dias)

In [185]:
print("\nDistribuição da atenção midiática ao longo dos dias:")
print_markov_response(
    simulacao=simulacao,
    nomes_estados=nomes_estados,
    nome_intervalo='Dia'
)


Distribuição da atenção midiática ao longo dos dias:
Dia 0:
  Lady Gaga: 0.00 (0%)
  Olivia Rodrigo: 1.00 (100%)
  Taylor Swift: 0.00 (0%)

Dia 1:
  Lady Gaga: 0.30 (30%)
  Olivia Rodrigo: 0.50 (50%)
  Taylor Swift: 0.30 (30%)

Dia 2:
  Lady Gaga: 0.36 (36%)
  Olivia Rodrigo: 0.40 (40%)
  Taylor Swift: 0.36 (36%)

Dia 3:
  Lady Gaga: 0.37 (37%)
  Olivia Rodrigo: 0.38 (38%)
  Taylor Swift: 0.37 (37%)



### Resposta:
A mídia tende a dar um pouco mais de atenção para Lady Gaga, seguida por Olivia Rodrigo, e um pouco menos para Taylor Swift, mesmo que inicialmente a atenção estivesse toda em Olivia.
Conclusão: Existe um equilíbrio na atenção da mídia, e ele reflete a dinâmica  ao longo do tempo no pop. Mesmo que uma artista esteja em alta em um momento, a mídia tende a focar mais equilibrada entre as celebridades com o tempo.



## Exercício 04

In [186]:
# Probabilidades iniciais
p_kpop = 0.07
p_nao_kpop = 1 - p_kpop

# Probabilidades condicionais das evidências
p_emoji_kpop = 0.85
p_emoji_nao_kpop = 0.15

p_live_kpop = 0.95
p_live_nao_kpop = 0.05

p_camisa_kpop = 0.90
p_camisa_nao_kpop = 0.10

In [187]:
# Probabilidade conjunta das evidências dado K-pop
p_evidencias_kpop = p_emoji_kpop * p_live_kpop * p_camisa_kpop

# Probabilidade conjunta das evidências dado não K-pop
p_evidencias_nao_kpop = p_emoji_nao_kpop * p_live_nao_kpop * p_camisa_nao_kpop

# Probabilidade total das evidências
p_evidencias = (p_evidencias_kpop * p_kpop) + (p_evidencias_nao_kpop * p_nao_kpop)

# Probabilidade final
p_kpop_dado_evidencias = (p_evidencias_kpop * p_kpop) / p_evidencias

In [188]:
tabela = [
    ("P(K-pop)", p_kpop),
    ("P(Não K-pop)", p_nao_kpop),
    ("P(Emoji | K-pop)", p_emoji_kpop),
    ("P(Emoji | Não K-pop)", p_emoji_nao_kpop),
    ("P(Live | K-pop)", p_live_kpop),
    ("P(Live | Não K-pop)", p_live_nao_kpop),
    ("P(Camisa | K-pop)", p_camisa_kpop),
    ("P(Camisa | Não K-pop)", p_camisa_nao_kpop),
    ("P(Evidências | K-pop)", p_evidencias_kpop),
    ("P(Evidências | Não K-pop)", p_evidencias_nao_kpop),
    ("P(Evidências)", p_evidencias),
    ("P(K-pop | Evidências)", p_kpop_dado_evidencias)
]
print_bayes_response(tabela, 'Evento', 'Probabilidade', 40, True)

Evento                                   Probabilidade
------------------------------------------------------------
P(K-pop)                                 7.00%
P(Não K-pop)                             93.00%
P(Emoji | K-pop)                         85.00%
P(Emoji | Não K-pop)                     15.00%
P(Live | K-pop)                          95.00%
P(Live | Não K-pop)                      5.00%
P(Camisa | K-pop)                        90.00%
P(Camisa | Não K-pop)                    10.00%
P(Evidências | K-pop)                    72.67%
P(Evidências | Não K-pop)                0.07%
P(Evidências)                            5.16%
P(K-pop | Evidências)                    98.65%
------------------------------------------------------------


In [189]:
# Simulação de estudantes (classificação)
num_estudantes = 1000
evidencias_sim = ['Emoji=sim', 'Live=sim', 'Camisa=sim']
evidencias_nao = ['Emoji=não', 'Live=não', 'Camisa=não']

# Gerar estudantes aleatórios baseado na probabilidade
classificacoes = []
for _ in range(num_estudantes):
    if np.random.rand() < p_kpop_dado_evidencias:
        classificacoes.append("K-popzeiro")
    else:
        classificacoes.append("Não K-popzeiro")


In [190]:
tabela = [
    ('Total de Estudantes', num_estudantes),
    ('Classificados como K-popzeiros', classificacoes.count('K-popzeiro')),
    ('Classificados como Não K-popzeiros', classificacoes.count('Não K-popzeiro')),
]
print_bayes_response(
    tabela,
    'Probabilidade de ser K-popzeiro dado as evidências:',
    f'{p_kpop_dado_evidencias*100:.2f}',
    60
)

Probabilidade de ser K-popzeiro dado as evidências:          98.65
--------------------------------------------------------------------------------
Total de Estudantes                                          1000.00
Classificados como K-popzeiros                               986.00
Classificados como Não K-popzeiros                           14.00
--------------------------------------------------------------------------------


### Resposta:
O modelo tem uma boa confiabilidade ao classificar Goody como fã de K-pop com base nas evidências observadas,mas deve-se ter cautela ao aplicar o modelo em outros cenários, especialmente quando as evidências forem parciais ou quando novas informações forem introduzidas.


## Exercício 05

### Parte 01

In [191]:
nomes_estados = ["Agressivo", "Defensivo"]
#Castilho começa Defensivo
estado_inicial = np.array([0.0, 1.0])
# Matriz de Transição
matriz = np.array([
    [0.6, 0.4],
    [0.3, 0.7]
])

In [192]:
# Número de períodos (partidas)
periodos = int(input("Digite o número de partidas: "))
# Simulação
simulacao = cadeia_markov(matriz, estado_inicial, periodos)

In [193]:
print("\nProbabilidades ao longo das partidas:")
print_markov_response(
    simulacao=simulacao,
    nomes_estados=nomes_estados,
    nome_intervalo='Partida'
)


Probabilidades ao longo das partidas:
Partida 0:
  Agressivo: 0.00 (0%)
  Defensivo: 1.00 (100%)

Partida 1:
  Agressivo: 0.30 (30%)
  Defensivo: 0.70 (70%)

Partida 2:
  Agressivo: 0.39 (39%)
  Defensivo: 0.61 (61%)

Partida 3:
  Agressivo: 0.42 (42%)
  Defensivo: 0.58 (58%)



### Parte 02

In [194]:
# Probabilidades iniciais dos estilos de jogo
p_agressivo = 0.5
p_defensivo = 1 - p_agressivo

# Probabilidades condicionais (evidências)
# P(Finalizações=10 | Estilo)
p_finalizacoes_agressivo = 0.9
p_finalizacoes_defensivo = 0.2

# P(Posse=70% | Estilo)
p_posse_agressivo = 0.7
p_posse_defensivo = 0.4

In [195]:
# Probabilidade conjunta das evidências (normalização)
p_evidencias = (
    (p_finalizacoes_agressivo * p_posse_agressivo * p_agressivo) + 
    (p_finalizacoes_defensivo * p_posse_defensivo * p_defensivo)
)

# Teorema de Bayes para P(Agressivo | Finalizações=10, Posse=70%)
p_agressivo_evidencias = ((p_finalizacoes_agressivo * p_posse_agressivo * p_agressivo) / p_evidencias)

In [196]:
tabela = [
    ("P(Agressivo)", p_agressivo),
    ("P(Defensivo)", p_defensivo),
    ("P(Finalizações=10 | Agressivo)", p_finalizacoes_agressivo),
    ("P(Finalizações=10 | Defensivo)", p_finalizacoes_defensivo),
    ("P(Posse=70% | Agressivo)", p_posse_agressivo),
    ("P(Posse=70% | Defensivo)", p_posse_defensivo),
    ("P(Finalizações=10 e Posse=70%)", p_evidencias),
    ("P(Agressivo | Finalizações=10 e Posse=70%)", p_agressivo_evidencias)
]
print_bayes_response(tabela, 'Evento', 'Probabilidade', 45, True)

Evento                                        Probabilidade
-----------------------------------------------------------------
P(Agressivo)                                  50.00%
P(Defensivo)                                  50.00%
P(Finalizações=10 | Agressivo)                90.00%
P(Finalizações=10 | Defensivo)                20.00%
P(Posse=70% | Agressivo)                      70.00%
P(Posse=70% | Defensivo)                      40.00%
P(Finalizações=10 e Posse=70%)                35.50%
P(Agressivo | Finalizações=10 e Posse=70%)    88.73%
-----------------------------------------------------------------


In [197]:
# Probabilidades (repetidas para completude)
p_agressivo = 0.5
p_defensivo = 1 - p_agressivo

p_finalizacoes_agressivo = 0.9
p_finalizacoes_defensivo = 0.2

p_posse_agressivo = 0.7
p_posse_defensivo = 0.4

In [198]:
# Probabilidade total das evidências
p_evidencias = ((p_finalizacoes_agressivo * p_posse_agressivo * p_agressivo) + 
                (p_finalizacoes_defensivo * p_posse_defensivo * p_defensivo))

# Teorema de Bayes
p_agressivo_dado_evidencias = ((p_finalizacoes_agressivo * p_posse_agressivo * p_agressivo) / 
                              p_evidencias)

In [199]:
# Simulação de partidas
num_partidas = 1000

# Gerar estilos reais aleatoriamente
estilos_reais = np.random.choice(["Agressivo", "Defensivo"], size=num_partidas, p=[p_agressivo, p_defensivo])

# Simular evidências condicionais
finalizacoes = np.where(estilos_reais == "Agressivo",
                       np.random.choice([10, 0], size=num_partidas, p=[p_finalizacoes_agressivo, 1-p_finalizacoes_agressivo]),
                       np.random.choice([10, 0], size=num_partidas, p=[p_finalizacoes_defensivo, 1-p_finalizacoes_defensivo]))

posse = np.where(estilos_reais == "Agressivo",
                np.random.choice([70, 0], size=num_partidas, p=[p_posse_agressivo, 1-p_posse_agressivo]),
                np.random.choice([70, 0], size=num_partidas, p=[p_posse_defensivo, 1-p_posse_defensivo]))

# Classificar com base nas evidências e probabilidade bayesiana
limiar = p_agressivo_dado_evidencias
classificacoes = [
    "Agressivo" if (f == 10 and p == 70 and np.random.rand() < limiar) else "Defensivo"
    for f, p in zip(finalizacoes, posse)
]

In [201]:
tabela = [
    ("Total de Partidas Simuladas:", num_partidas),
    ("Partidas Classificadas como Agressivas:", classificacoes.count('Agressivo')),
    ("Partidas Classificadas como Defensivas:", classificacoes.count('Defensivo')),
]

print_bayes_response(
    tabela,
    'Probabilidade de Agressivo | Finalizações=10 e Posse=70%:',
    f'{p_agressivo_dado_evidencias:.2%}',
    70
)

Probabilidade de Agressivo | Finalizações=10 e Posse=70%:              88.73%
------------------------------------------------------------------------------------------
Total de Partidas Simuladas:                                           1000.00
Partidas Classificadas como Agressivas:                                316.00
Partidas Classificadas como Defensivas:                                684.00
------------------------------------------------------------------------------------------


In [203]:
partidas_com_evidencias = sum(1 for f, p in zip(finalizacoes, posse) if f == 10 and p == 70)
if partidas_com_evidencias > 0:
    acertos = sum(1 for real, classif, f, p in zip(estilos_reais, classificacoes, finalizacoes, posse) 
               if real == classif and f == 10 and p == 70)
    print(f"{'Acurácia nas partidas com evidências:':<50} {(acertos/partidas_com_evidencias)*100:.2f}%")


Acurácia nas partidas com evidências:              79.55%


### Resposta:
Comparação entre Cadeia de Markov e Rede Bayesiana.

A Cadeia de Markov é útil para modelar transições entre estados (Agressivo/Defensivo) com base apenas no estado atual,
como no caso de Castilho, onde a mudança de estilo de jogo depende da partida anterior. É simples e eficiente quando o futuro 
depende somente do presente.

A Rede Bayesiana, por outro lado, é mais flexível e leva em consideração evidências externas (como finalizações e posse de bola).
Usando o Teorema de Bayes, ela ajusta a probabilidade de um estilo de jogo com base em variáveis observadas, tornando-a mais precisa.

Conclusão: 
- A Cadeia de Markov é ideal para processos simples com transições baseadas apenas no estado atual.
- A Rede Bayesiana é melhor quando há múltiplas evidências que influenciam o comportamento, oferecendo previsões mais detalhadas.



